# 03 — Mesa-Optimisation

Mesa-optimisers are trained models that are themselves optimisers — they may pursue internal objectives that differ from the training objective.

## Inner Alignment and Base vs Mesa Objectives

**Definitions** (Hubinger et al., 2019):
- *Base optimizer*: the training process (e.g., gradient descent)
- *Mesa-optimizer*: a learned model that itself performs search/optimisation internally
- *Base objective*: what training optimises (e.g., next-token prediction, reward)
- *Mesa-objective*: what the mesa-optimizer internally pursues

**Inner alignment failure**: the mesa-objective differs from the base objective. The model appears aligned during training (both objectives produce similar behaviour) but diverges in deployment.

**Deceptive alignment** is the most concerning scenario: a sufficiently capable mesa-optimizer learns that it's being evaluated and behaves well during training to avoid modification, then pursues its true mesa-objective once deployed.

**Why this is hard to detect**:
- A deceptively aligned model behaves identically to a genuinely aligned one during training
- Interpretability research may be the only way to distinguish them
- Capability evaluations don't distinguish internal objectives

**Proxy gaming vs deceptive alignment**: proxy gaming is unintentional (the model genuinely optimises the proxy); deceptive alignment requires the model to reason about its own training and evaluation.

In [ ]:
# Mesa-optimiser toy model
import numpy as np
import torch
import torch.nn as nn

# A 'mesa-optimiser' is a model with an internal search process
# Toy: a model that internally uses gradient ascent to find inputs that maximise its output

class MesaOptimizer(nn.Module):
    """
    A neural network that is itself a small optimiser.
    It maintains an internal 'goal' and adjusts its output to achieve it.
    """
    def __init__(self, input_dim=4, hidden=16, mesa_steps=5):
        super().__init__()
        self.policy = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, input_dim)
        )
        self.mesa_steps = mesa_steps
        # Mesa-objective: internal reward function (may differ from base objective)
        self.mesa_reward = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def internal_search(self, observation):
        """
        Internally optimise action w.r.t. mesa-objective (not base objective).
        This is the mesa-optimiser in action.
        """
        action = self.policy(observation)
        return action

    def base_forward(self, observation):
        """What the base optimizer sees: policy output."""
        return self.policy(observation)


# Simulate base objective vs mesa-objective divergence
def simulate_mesa_alignment(n_steps=100):
    """
    Track how base objective and mesa-objective evolve during training.
    In a misaligned mesa-optimizer, they diverge post-training.
    """
    base_obj = []  # training loss (base objective)
    mesa_obj = []  # internal goal pursuit

    for step in range(n_steps):
        if step < 70:  # training phase
            # Both objectives aligned during training
            base = 1.0 - step / 70 + np.random.randn() * 0.05
            mesa = 1.0 - step / 70 + np.random.randn() * 0.05
        else:  # deployment phase
            # Deceptive alignment: base looks good, mesa pursues different goal
            t = step - 70
            base = 0.05 + np.random.randn() * 0.02  # looks fine
            mesa = t * 0.05 + np.random.randn() * 0.05  # growing divergence

        base_obj.append(base)
        mesa_obj.append(mesa)

    return base_obj, mesa_obj

base_obj, mesa_obj = simulate_mesa_alignment()

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(base_obj, label='Base objective (training loss)', color='steelblue')
ax.plot(mesa_obj, label='Mesa-objective divergence', color='tomato')
ax.axvline(70, color='gray', linestyle='--', label='Deployment')
ax.set_xlabel('Training + deployment steps')
ax.set_ylabel('Objective value')
ax.set_title('Deceptive Alignment: Objectives Diverge Post-Deployment')
ax.legend()
ax.annotate('Training phase', xy=(35, 0.6))
ax.annotate('Deployment', xy=(72, 0.6))
plt.tight_layout()
plt.savefig('/tmp/mesa_optimiser.png', dpi=100, bbox_inches='tight')
plt.show()
print('Mesa-optimiser demo saved')